# 存在偏航运动：第一类与无乘子第二类拉格朗日方程
$\psi$ 与偏航速度不作小量；只把 $\theta,p=\dot\theta$ 作为俯仰小扰动。
本分支车轮轴对称且左右相同，$m_W=m_L=m_R$，$J_s=I_{Lyy}=I_{Ryy}$，$J_t=I_{Lxx}=I_{Lzz}=I_{Rxx}=I_{Rzz}$。
先由共用速度构造精确能量并求导，再展开小俯仰方程，避免丢失 $\theta\omega^2$。
可允许速度为 $u=[v,\omega,p]^T$，$v=\dot l$；$l$ 是历史版本的累计前向行程。
$x,y$ 保持世界位置，$\dot x=v\cos\psi,\dot y=v\sin\psi$，不把 $v$ 冒充世界 $\dot x$。


In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'base.py').exists():
    if root==root.parent: raise RuntimeError('找不到 theory/base.py')
    root=root.parent
sys.path.insert(0,str(root))
import sympy as s
import numpy as np
from IPython.display import display, Math
# 只改 LaTeX 显示，不替换计算符号：dx -> x 上点，ddx -> x 双点。
from base import q, qd, qdd
latex_names = {}
for coordinate, velocity, acceleration in zip(q, qd, qdd):
    name = s.latex(coordinate)
    latex_names[velocity] = r'\dot{' + name + '}'
    latex_names[acceleration] = r'\ddot{' + name + '}'
s.init_printing(symbol_names=latex_names)
np.set_printoptions(precision=8,suppress=True,linewidth=130)
def show(label, expression):
    display(Math(label+' = '+s.latex(expression, symbol_names=latex_names)))


In [2]:
from turning_balance.lagrange import *
from lagrange_utils import tau
m=build()

## 从共享运动学到本场景能量

In [3]:
show('T',m['T']);show('V',m['V'])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 第一类：六个广义方向与三个约束反力
坐标顺序 $q=[x,y,\psi,\theta,\theta_L,\theta_R]^T$。

$$M_6\ddot q+h_6=B_6\tau+A_c^T\lambda,\quad A_c\dot q=0,$$
$$\begin{bmatrix}M_6&-A_c^T\\ A_c&0\end{bmatrix}
\begin{bmatrix}\ddot q\\\lambda\end{bmatrix}=
\begin{bmatrix}B_6\tau-h_6\\-\dot A_c\dot q\end{bmatrix}.$$

电机反作用力矩同时作用于车体：$Q_\theta=-(\tau_L+\tau_R)$，不能漏掉。
下面列出全部分块矩阵、偏置和最终9×9系数矩阵；$\lambda$ 的数值依赖约束行的归一化。

In [4]:
for key in ('M6','h6','B6','Acon','augmented_matrix','augmented_rhs'):show(key.replace('_',r'\_'),m[key])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 第二类：消除乘子，保留非完整约束的修正项
选取可允许独立速度 $u$，使 $\dot q=S(q)u$、$A_c S=0$。
令 $p_q=\partial L/\partial\dot q$（这里的 $p_q$ 是动量，不是俯仰角速度 p）。

$$\frac{d}{dt}(S^Tp_q)-S^T\frac{\partial L}{\partial q}-\dot S^Tp_q=S^TB_6\tau.$$

这就是无乘子的拉格朗日—达朗贝尔方程，本文称“第二类形式”。它**不等于**先把速度约束代入 L 再直接对三个假想全局坐标套普通 Euler–Lagrange 算子。

$$M=S^TM_6S,\quad h=S^T(M_6\dot S u+h_6),\quad B_\tau=S^TB_6,$$
$$M\dot u+h=B_\tau\tau.$$

代码还从上面的动量公式独立构造 M_second、h_second，并与投影式逐项比较。

In [5]:
show('S',m['S']);show('u',m['u'])
for key in ('M','h','B'):show(key,m[key])
assert clean(m['M']-m['M_second'])==s.zeros(3)
assert clean(m['h']-m['h_second'])==s.zeros(3,1)
print('PASS：两种拉格朗日形式严格一致')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS：两种拉格朗日形式严格一致


## 只对俯仰作一阶展开
令 $(\theta,p)\mapsto\epsilon(\theta,p)$；保留 $O(\epsilon)$，$v,\omega$ 不缩放。
这里指局部运动附近的加速度有界展开，不适用于俯仰加速度随 $1/\epsilon$ 发散的情况。

$$M_{small}\dot u+h_{small}=B_\tau\tau.$$

$h_{small}$ 含速度乘积，系统仍是非线性的，不能给任意转向运动使用一组固定世界坐标 A、B。
五个辅助系数依次是 $a,c,e,J,D$；$J$ 在此是偏航惯量，LQR 章节的 J 才是代价泛函。

In [6]:
show(s.latex(s.Matrix(s.symbols('a c e J D'))),m['coefficients'])
show('M_{small}',m['M_small']);show('h_{small}',m['h_small'])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>